## 1. Configuração e leitura

Carregamos apenas os arquivos com sufixo `_13cat`. Cada dataset vira uma entrada no
dicionário `datasets`, identificada pelo nome da área.

As constantes de cor ficam no topo para facilitar ajuste. Uma cor única para todas as
caixas é proposital: cada painel mostra uma coluna diferente, e a identidade dela já
está no título — variar a cor sem que ela signifique nada só adicionaria ruído.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S3"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NCOLS = 4                # subplots por linha
MULT_EXTREMO = 2.5       # multiplo do IQR que define outlier extremo
SALVAR = False           # True -> grava os PNGs na pasta boxplots/

COR_CAIXA = "#5B7FA6"    # azul dessaturado, mesma cor para todas as caixas
COR_MEDIANA = "#1F3247"
COR_MODERADO = "#8A8F98"
COR_EXTREMO = "#D97706"

arquivos = sorted(glob.glob("dataset_*_13cat.csv"))
print("Datasets encontrados:", len(arquivos))

datasets = {}
for arq in arquivos:
    nome = arq.replace("dataset_", "").replace("_13cat.csv", "")
    datasets[nome] = pd.read_csv(arq)
    print(f"  {nome:16s} -> {datasets[nome].shape}")

## 2. Identificar as colunas com nulos

Selecionamos as colunas **numéricas** que ainda têm ao menos um valor nulo. O filtro por
tipo evita que colunas categóricas ou dummies entrem na lista — boxplot só faz sentido
para variável contínua.

In [ ]:
def colunas_com_nulos(df):
    """Colunas numéricas que possuem ao menos um valor nulo."""
    return [c for c in df.columns
            if df[c].isna().any() and pd.api.types.is_numeric_dtype(df[c])]


alvo = {nome: colunas_com_nulos(df) for nome, df in datasets.items()}

total = 0
for nome, cols in alvo.items():
    if cols:
        print(f"{nome:16s} {len(cols):>2} coluna(s): {', '.join(cols)}")
        total += len(cols)
    else:
        print(f"{nome:16s}  0 coluna(s) -- sem nulos, será pulado")

print(f"\nTotal de colunas a plotar: {total}")

## 3. Classificar os outliers

A regra de Tukey usa a distância até os quartis em múltiplos do IQR (`Q3 - Q1`):

| Faixa | Cerca | Leitura |
|---|---|---|
| **Moderado** | entre 1,5x e 2,5x o IQR | desvio gradual, cauda da distribuição |
| **Extremo** | >= 2,5x o IQR | ponto destacado do corpo dos dados |

A separação importa na hora de imputar: uma coluna com IQR pequeno e muitos pontos
**extremos** costuma indicar dois regimes distintos de operação (bimodalidade), e não
erro de medição — nesse caso, preencher com a mediana global mistura os dois grupos.
Já uma cauda de pontos **moderados** é variação normal.

In [ ]:
def classifica_outliers(serie):
    """Separa os valores não-nulos em moderados e extremos, pelo corte MULT_EXTREMO."""
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    extremos = s[(s <= q1 - MULT_EXTREMO * iqr) | (s >= q3 + MULT_EXTREMO * iqr)]
    fora = s[(s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)]
    moderados = fora.drop(extremos.index)   # extremos são subconjunto dos que estão fora
    return moderados, extremos, iqr

## 4. Plotar

Uma figura por dataset, um subplot por coluna. Cada painel tem **escala própria**: as
faixas de temperatura variam muito entre sensores, e uma escala compartilhada achataria
justamente as colunas de IQR pequeno, que são as mais interessantes.

Os pontos fora das cercas são desenhados à mão (`showfliers=False` no boxplot) para
poder colori-los pelas duas faixas da seção anterior. O jitter horizontal evita que
valores próximos se empilhem e escondam uns aos outros.

In [ ]:
def plota_dataset(nome, df, cols):
    """Desenha uma figura com um boxplot por coluna com nulos."""
    if not cols:
        print(f"{nome}: sem colunas com nulos, pulando")
        return

    nrows = -(-len(cols) // NCOLS)   # divisão com arredondamento para cima
    fig, axes = plt.subplots(nrows, NCOLS, figsize=(3.2 * NCOLS, 3.6 * nrows))
    axes = np.atleast_1d(axes).ravel()

    rng = np.random.default_rng(0)   # semente fixa: o jitter não muda entre execuções

    for ax, col in zip(axes, cols):
        s = df[col].dropna()
        moderados, extremos, iqr = classifica_outliers(df[col])

        ax.boxplot(
            s, widths=0.45, showfliers=False, patch_artist=True,
            boxprops=dict(facecolor=COR_CAIXA, edgecolor=COR_MEDIANA,
                          linewidth=1.2, alpha=0.55),
            medianprops=dict(color=COR_MEDIANA, linewidth=2),
            whiskerprops=dict(color=COR_MEDIANA, linewidth=1.2),
            capprops=dict(color=COR_MEDIANA, linewidth=1.2),
        )

        for valores, cor in ((moderados, COR_MODERADO), (extremos, COR_EXTREMO)):
            if len(valores):
                x = 1 + rng.uniform(-0.06, 0.06, len(valores))
                ax.scatter(x, valores, s=28, color=cor, edgecolor="white",
                           linewidth=0.8, zorder=3)

        n_nulos = int(df[col].isna().sum())
        n_out = len(moderados) + len(extremos)
        titulo = col if len(col) <= 32 else col[:30] + "..."
        ax.set_title(titulo, fontsize=9.5, pad=8)
        ax.set_xlabel(
            f"{n_nulos} {'nulo' if n_nulos == 1 else 'nulos'}  |  IQR {iqr:.2f}  |  "
            f"{n_out} {'outlier' if n_out == 1 else 'outliers'}",
            fontsize=8, color="#6B7280",
        )
        ax.set_ylabel("°C", fontsize=8)
        ax.set_xticks([])
        ax.grid(axis="y", color="#E5E7EB", linewidth=0.8)
        ax.set_axisbelow(True)
        for lado in ("top", "right", "bottom"):
            ax.spines[lado].set_visible(False)
        ax.spines["left"].set_color("#D1D5DB")

    for ax in axes[len(cols):]:
        ax.set_visible(False)

    # uma única legenda para a figura toda, em vez de repetir em cada subplot
    mult = f"{MULT_EXTREMO}".replace(".", ",")   # 2.5 -> 2,5 no rotulo
    marcadores = [
        plt.Line2D([], [], marker="o", linestyle="", color=COR_MODERADO,
                   markeredgecolor="white", markersize=7,
                   label=f"outlier moderado (1,5x-{mult}x IQR)"),
        plt.Line2D([], [], marker="o", linestyle="", color=COR_EXTREMO,
                   markeredgecolor="white", markersize=7,
                   label=f"outlier extremo (>= {mult}x IQR)"),
    ]
    fig.legend(handles=marcadores, fontsize=9, frameon=False,
               loc="upper right", bbox_to_anchor=(0.995, 0.995))

    fig.suptitle(f"{nome} — colunas com valores nulos",
                 fontsize=13, x=0.02, y=0.995, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.97))

    if SALVAR:
        import os
        os.makedirs("boxplots", exist_ok=True)
        destino = os.path.join("boxplots", f"boxplot_{nome}_13cat.png")
        fig.savefig(destino, dpi=150, bbox_inches="tight", facecolor="white")
        print("salvo:", destino)

    plt.show()

In [ ]:
for nome, df in datasets.items():
    plota_dataset(nome, df, alvo[nome])

## 5. Tabela resumo

Os mesmos números dos rodapés, reunidos para consulta. Ordenada pelo total de outliers,
para que as colunas mais dispersas apareçam primeiro.

In [ ]:
linhas = []
for nome, df in datasets.items():
    for col in alvo[nome]:
        moderados, extremos, iqr = classifica_outliers(df[col])
        n_nulos = int(df[col].isna().sum())
        linhas.append({
            "dataset": nome,
            "coluna": col,
            "nulos": n_nulos,
            "% nulos": round(100 * n_nulos / len(df), 1),
            "IQR": round(iqr, 2),
            "moderados": len(moderados),
            "extremos": len(extremos),
            "total outliers": len(moderados) + len(extremos),
        })

resumo = pd.DataFrame(linhas).sort_values("total outliers", ascending=False)
print(f"{len(resumo)} colunas com nulos  |  "
      f"{resumo['total outliers'].sum()} outliers "
      f"({resumo['extremos'].sum()} extremos)")
resumo.reset_index(drop=True)